In [9]:
# ==============================================================================
# RHONN SYSTEM IDENTIFICATION: EKF vs PF COMPARISON
# Extended Kalman Filter vs Particle Filter for RHONN Weight Learning
# ==============================================================================

import numpy as np
import plotly.graph_objects as go

print("Libraries imported successfully")

Libraries imported successfully


In [10]:
# ==============================================================================
# LORENZ SYSTEM DYNAMICS AND PLANT MODEL
# ==============================================================================

def plant_dynamics(x, u):
    """Lorenz system dynamics: dx/dt = f(x,u)"""
    sigma, rho, beta = 10.0, 28.0, 8.0/3.0
    x_dot = sigma * (x[1] - x[0])
    y_dot = x[0] * (rho - x[2]) - x[1]
    z_dot = x[0] * x[1] - beta * x[2]
    return np.array([x_dot, y_dot, z_dot])

def plant(x_k, u_k, dt=0.01, process_noise_type='laplacian', process_noise_std=1e-3):
    """One Euler step of the discrete plant with process noise"""
    x_dot = plant_dynamics(x_k, u_k)
    x_kp1 = x_k + dt * x_dot

    # Add process noise
    if process_noise_type == 'laplacian':
        noise = np.random.laplace(0, process_noise_std, size=x_kp1.shape)
    elif process_noise_type == 'uniform':
        a = np.sqrt(3) * process_noise_std
        noise = np.random.uniform(-a, a, size=x_kp1.shape)
    else:  # gaussian
        noise = np.random.normal(0, process_noise_std, size=x_kp1.shape)

    return x_kp1 + noise

print("Lorenz system dynamics defined")

Lorenz system dynamics defined


In [11]:
# ==============================================================================
# RHONN STRUCTURE AND FEATURE ENGINEERING
# ==============================================================================

def sigmoidal(z, beta=1.0):
    """Sigmoid activation function S(z) = 1/(1 + exp(-beta*z))"""
    return 1.0 / (1.0 + np.exp(-beta * z))

def construct_z_vector(x_est):
    """
    Construct 10-dimensional feature vector for 3-state Lorenz system:
    z = [S(x1)^3, S(x2), S(x3), S(x1)S(x2), S(x1)S(x3), S(x2)S(x3), 
         S(x1)^2, S(x2)^2, S(x3)^2, 1]
    """
    s_x1 = sigmoidal(x_est[0])  # x
    s_x2 = sigmoidal(x_est[1])  # y
    s_x3 = sigmoidal(x_est[2])  # z
    
    return np.array([
        s_x1**3, s_x2, s_x3,                    # Modified linear terms
        s_x1*s_x2, s_x1*s_x3, s_x2*s_x3,       # Cross terms
        s_x1**2, s_x2**2, s_x3**2,             # Quadratic terms
        1.0                                      # Bias
    ])

def RHONN_predict(x_state_for_z, w_neuron):
    """
    Predict next-state component: x_i(k+1) = w_i^T * z(x(k))
    """
    z_i = construct_z_vector(x_state_for_z)
    if len(z_i) != len(w_neuron):
        raise ValueError(f"Dimension mismatch: z({len(z_i)}) vs w({len(w_neuron)})")
    return np.dot(w_neuron, z_i)

print("RHONN structure defined")

RHONN structure defined


In [12]:
# ==============================================================================
# EXTENDED KALMAN FILTER RHONN TRAINER
# ==============================================================================

class EKF_RHONN_Trainer:
    """Extended Kalman Filter for online RHONN weight learning"""
    
    def __init__(self, num_neurons, num_weights_per_neuron, initial_weights=None,
                 Q_init=1e-3, R_init=1e-3, P_init=1.0, eta=0.9):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.Q = Q_init * np.eye(num_weights_per_neuron)  # Process noise
        self.R = R_init                                   # Measurement noise
        self.eta = eta                                    # Forgetting factor

        # Initialize weights and covariance matrices
        if initial_weights is None:
            self.weights = [np.random.uniform(-0.1, 0.1, num_weights_per_neuron) 
                           for _ in range(num_neurons)]
        else:
            self.weights = [np.copy(w) for w in initial_weights]
        
        self.P = [P_init * np.eye(num_weights_per_neuron) for _ in range(num_neurons)]

    def update(self, chi_kp1, x_hat_previous):
        """Update weights using EKF for each neuron"""
        for i in range(self.num_neurons):
            # Build feature vector from previous estimate
            z_i = construct_z_vector(x_hat_previous)
            
            # Predicted output
            y_pred = np.dot(self.weights[i], z_i)
            
            # Prediction covariance update
            self.P[i] = (1/self.eta) * self.P[i] + self.Q
            
            # Innovation and Kalman gain
            H = z_i.reshape(1, -1)  # Jacobian (linear in weights)
            PHT = self.P[i] @ H.T
            S = H @ PHT + self.R
            K = PHT / S
            
            # Weight and covariance update
            innovation = chi_kp1[i] - y_pred
            self.weights[i] += K.flatten() * innovation
            self.P[i] = (np.eye(self.num_weights_per_neuron) - K @ H) @ self.P[i]

print("EKF trainer defined")

EKF trainer defined


In [13]:
# ==============================================================================
# PARTICLE SWARM OPTIMIZATION FOR PF PARAMETER TUNING
# ==============================================================================

class PSO_PF_Optimizer:
    """Particle Swarm Optimization for PF parameter tuning"""
    
    def __init__(self, n_particles=30, n_iterations=20, bounds=None):
        self.n_particles = n_particles
        self.n_iterations = n_iterations
        self.bounds = bounds or {'Q_std': (0.01, 1.0), 'R_std': (0.001, 0.1)}
        
        # PSO parameters
        self.w = 0.5      # Inertia weight
        self.c1 = 1.5     # Cognitive parameter
        self.c2 = 1.5     # Social parameter
        
        # Initialize swarm
        self._initialize_swarm()
        
    def _initialize_swarm(self):
        """Initialize particle positions and velocities"""
        self.positions = np.random.uniform(
            [self.bounds['Q_std'][0], self.bounds['R_std'][0]],
            [self.bounds['Q_std'][1], self.bounds['R_std'][1]],
            (self.n_particles, 2)
        )
        self.velocities = np.random.uniform(-0.1, 0.1, (self.n_particles, 2))
        self.best_positions = self.positions.copy()
        self.best_scores = np.full(self.n_particles, np.inf)
        self.global_best_position = None
        self.global_best_score = np.inf
        
    def _evaluate_fitness(self, Q_std, R_std, num_neurons=3, n_steps=200, n_particles_pf=100):
        """Evaluate fitness function for given parameters"""
        try:
            # Create temporary PF trainer
            temp_trainer = PF_RHONN_Trainer(
                num_neurons, 10, n_particles=n_particles_pf,
                Q_std=Q_std, R_std=R_std, ess_threshold=n_particles_pf//4
            )
            
            # Run short simulation
            x_true_temp = np.zeros((n_steps, 3))
            x_hat_temp = np.zeros((n_steps, 3))
            x_true_temp[0] = [1.0, 1.0, 1.0]
            x_hat_temp[0] = x_true_temp[0]
            
            total_error = 0
            for k in range(n_steps-1):
                x_true_temp[k+1] = plant(x_true_temp[k], 0.0, 0.01)
                temp_trainer.update(x_true_temp[k+1], x_hat_temp[k])
                weights_est = temp_trainer.get_estimate()
                
                for i in range(3):
                    x_hat_temp[k+1, i] = RHONN_predict(x_hat_temp[k], weights_est[i])
                
                total_error += np.mean((x_true_temp[k+1] - x_hat_temp[k+1])**2)
            
            return total_error / n_steps
            
        except Exception as e:
            return 1e6  # Return high penalty for invalid parameters
    
    def optimize(self):
        """Run PSO optimization"""
        history = []
        
        for iteration in range(self.n_iterations):
            # Evaluate all particles
            for i in range(self.n_particles):
                Q_std, R_std = self.positions[i]
                score = self._evaluate_fitness(Q_std, R_std)
                
                # Update personal best
                if score < self.best_scores[i]:
                    self.best_scores[i] = score
                    self.best_positions[i] = self.positions[i].copy()
                
                # Update global best
                if score < self.global_best_score:
                    self.global_best_score = score
                    self.global_best_position = self.positions[i].copy()
            
            # Update velocities and positions
            for i in range(self.n_particles):
                r1, r2 = np.random.random(2), np.random.random(2)
                
                # Velocity update
                cognitive = self.c1 * r1 * (self.best_positions[i] - self.positions[i])
                social = self.c2 * r2 * (self.global_best_position - self.positions[i])
                self.velocities[i] = (self.w * self.velocities[i] + cognitive + social)
                
                # Position update with bounds checking
                self.positions[i] += self.velocities[i]
                self.positions[i] = np.clip(self.positions[i], 
                                          [self.bounds['Q_std'][0], self.bounds['R_std'][0]],
                                          [self.bounds['Q_std'][1], self.bounds['R_std'][1]])
            
            # Store history
            avg_score = np.mean(self.best_scores)
            history.append({'iteration': iteration, 'best_score': self.global_best_score, 'avg_score': avg_score})
            
            if iteration % 5 == 0:
                print(f"PSO Iteration {iteration}: Best={self.global_best_score:.6f}, Avg={avg_score:.6f}")
        
        return {
            'Q_std': self.global_best_position[0],
            'R_std': self.global_best_position[1],
            'score': self.global_best_score,
            'history': history
        }

print("PSO optimizer defined")

PSO optimizer defined


In [ ]:
class PF_RHONN_Trainer:
    """
    Particle filter over neuron weights (per neuron).
    - Predict (random walk on weights with adaptive noise)
    - Update (likelihood from chi_{k+1} vs prediction built with z from estimated states)
    - ESS-triggered resampling with better numerical stability
    - Adaptive Q_std and R_std similar to EKF forgetting factor
    """
    def __init__(self, num_neurons, num_weights_per_neuron, n_particles=200,
                 initial_weights=None, Q_std=0.02, R_std=0.05, ess_threshold=None,
                 adaptation_rate=0.95, min_Q_std=1e-4, max_Q_std=2.0, 
                 min_R_std=1e-3, max_R_std=1.0,
                 # >>> NEW baseline knobs:
                 robust='laplace',          # 'laplace' or 'gaussian'
                 temperature=0.7,           # likelihood tempering τ in (0,1]
                 regularize_scale=2e-3      # jitter after any resample
                 ):
        self.num_neurons = num_neurons
        self.num_weights_per_neuron = num_weights_per_neuron
        self.n_particles = n_particles
        
        # Adaptive noise parameters
        self.Q_std = Q_std
        self.R_std = R_std
        self.R_var = R_std**2
        self.adaptation_rate = adaptation_rate  # Similar to EKF eta parameter
        self.min_Q_std = min_Q_std
        self.max_Q_std = max_Q_std
        self.min_R_std = min_R_std
        self.max_R_std = max_R_std

                # --- NEW: robust likelihood + tempering + regularization
        self.robust = robust
        self.temperature = float(temperature)
        self.regularize_scale = float(regularize_scale)

        # --- NEW: per-neuron Q (predict noise) so each output adapts independently
        self.per_neuron_Q = np.full(self.num_neurons, float(Q_std))

        # (keep your existing self.Q_std for reporting/backward-compat, but it
        # won't be used in the predict step anymore; we’ll report avg of per_neuron_Q)

        
        # Track innovation statistics for adaptation
        self.innovation_history = []
        self.window_size = 50  # Window for adaptation
        
        self.ess_threshold = ess_threshold if ess_threshold is not None else n_particles / 2.0

        self.particles = []
        self.weights_pf = []

        for i in range(num_neurons):
            if initial_weights is not None and i < len(initial_weights):
                base = np.copy(initial_weights[i])
                # Add more spread for better exploration
                particles_i = base + np.random.randn(n_particles, num_weights_per_neuron) * 0.2
            else:
                particles_i = np.random.randn(n_particles, num_weights_per_neuron) * 0.2
            self.particles.append(particles_i)
            self.weights_pf.append(np.ones(n_particles) / n_particles)

    def _ess(self, w):
        w = w / np.sum(w)
        return 1.0 / np.sum(w**2)

    def _resample_systematic(self, neuron_index):
        w = self.weights_pf[neuron_index]
        p = self.particles[neuron_index]

        w = w / np.sum(w)
        N = len(w)
        u0 = np.random.uniform(0.0, 1.0 / N)
        cdf = np.cumsum(w)

        indexes = np.zeros(N, dtype=int)
        i, j = 0, 0
        while i < N:
            u = u0 + i / N
            while u > cdf[j]:
                j += 1
            indexes[i] = j
            i += 1

        self.particles[neuron_index] = p[indexes]
        self.weights_pf[neuron_index] = np.ones(N) / N
        # >>> NEW: small jitter to fight impoverishment
        self._regularize_after_resample(neuron_index)

    def _adapt_noise_parameters(self, innovations):
        """
        Adapt Q (per neuron) and R (global) based on recent innovation stats.
        `innovations` is a list of length num_neurons with mean innovation (per neuron) for this step.
        """
        # --- Track recent scalar innovations for R adaptation (global)
        self.innovation_history.extend(innovations)
        if len(self.innovation_history) > self.window_size:
            self.innovation_history = self.innovation_history[-self.window_size:]

        # --- Per-neuron Q adaptation using the per-neuron mean |innovation|
        for i, m_innov in enumerate(innovations):
            abs_m = abs(float(m_innov))
            target_q = self.per_neuron_Q[i]

            # Heuristic: if mean |innov| is large relative to R_std, explore more
            if abs_m > 2.0 * self.R_std:
                target_q = min(self.max_Q_std, self.per_neuron_Q[i] * (1.0 + 0.1 * (1 - self.adaptation_rate)))
            elif abs_m < 0.5 * self.R_std:
                target_q = max(self.min_Q_std, self.per_neuron_Q[i] * self.adaptation_rate)

            # Momentum smoothing
            self.per_neuron_Q[i] = 0.9 * self.per_neuron_Q[i] + 0.1 * target_q

        # --- Global R adaptation (your original idea, smoothed)
        if len(self.innovation_history) >= 10:
            innovations_array = np.array(self.innovation_history, dtype=float)
            innovation_variance = np.var(innovations_array)
            target_R_var = max(innovation_variance * 0.5, self.min_R_std ** 2)
            target_R_std = min(np.sqrt(target_R_var), self.max_R_std)

            # Smooth adaptation for R
            self.R_std = self.adaptation_rate * self.R_std + (1 - self.adaptation_rate) * target_R_std
            self.R_var = self.R_std ** 2


    def update(self, chi_kp1, x_hat_previous):
        """
        One PF step over all neuron weight-sets with adaptive noise parameters.
        chi_kp1: measured true states at k+1 (targets) - in real scenario, this would be measurements
        x_hat_previous: previous estimate at k (to complete z) - only this is used for z
        """
        # Build z from estimated states (no cheating!)
        z = construct_z_vector(x_hat_previous)  # (num_features,)

        # Collect innovations for adaptation
        step_innovations = []

        # 1) Predict: random walk on weights with adaptive noise
        for i in range(self.num_neurons):
            q_i = self.per_neuron_Q[i]
            self.particles[i] += np.random.randn(self.n_particles, self.num_weights_per_neuron) * q_i

        # 2) Update: importance weights with Gaussian likelihood
        for i in range(self.num_neurons):
            w_mat = self.particles[i]                               # (N, num_features)
            x_pred_particles = w_mat @ z                            # (N,)
            innov = chi_kp1[i] - x_pred_particles

            # Store weighted mean innovation for adaptation
            weights_norm = self.weights_pf[i] / np.sum(self.weights_pf[i])
            mean_innovation = np.sum(weights_norm * innov)
            step_innovations.append(mean_innovation)

            # Robust log-likelihood (Laplace or Gaussian) + stabilization + tempering
            log_likelihood = self._log_like(innov)
            log_likelihood -= np.max(log_likelihood)  # stabilize
            likelihood = np.exp(log_likelihood)
            # Tempering (τ in (0,1]) flattens too-peaky likelihoods
            if self.temperature is not None and self.temperature > 0.0:
                likelihood = likelihood ** self.temperature

            # Clip to avoid denormals
            likelihood = np.maximum(likelihood, 1e-12)

            # Update particle weights
            self.weights_pf[i] *= likelihood
            
            # Normalize weights
            weight_sum = np.sum(self.weights_pf[i])
            if weight_sum < 1e-300:
                # Reset weights if they collapse
                self.weights_pf[i] = np.ones(self.n_particles) / self.n_particles
            else:
                self.weights_pf[i] /= weight_sum

            # Resample if ESS is low
            if self._ess(self.weights_pf[i]) < self.ess_threshold:
                self._resample_systematic(i)
        
        # 3) Adapt noise parameters based on innovations
        self._adapt_noise_parameters(step_innovations)


    def _regularize_after_resample(self, neuron_index):
        if self.regularize_scale and self.regularize_scale > 0.0:
            self.particles[neuron_index] += (
                np.random.randn(self.n_particles, self.num_weights_per_neuron) * self.regularize_scale
            )


    def _log_like(self, innov):
        """
        Return elementwise log-likelihood up to a constant (array same shape as innov).
        - Laplace: b chosen so that Var(Laplace)=2b^2 = R_var
        - Gaussian: standard quadratic form
        """
        if self.robust == 'laplace':
            b = np.sqrt(self.R_var / 2.0) + 1e-12
            return -np.abs(innov) / b
        else:
            return -0.5 * (innov ** 2) / (self.R_var + 1e-12)


    def get_estimate(self):
        """Mean of particles per neuron (after any resampling)."""
        estimates = []
        for i in range(self.num_neurons):
            # Use weighted mean instead of simple mean
            weights = self.weights_pf[i]
            particles = self.particles[i]
            estimate = np.average(particles, axis=0, weights=weights)
            estimates.append(estimate)
        return estimates

    def get_variance(self):
        """Get variance of particles per neuron for uncertainty quantification."""
        variances = []
        for i in range(self.num_neurons):
            weights = self.weights_pf[i]
            particles = self.particles[i]
            mean = np.average(particles, axis=0, weights=weights)
            # Weighted variance
            variance = np.average((particles - mean)**2, axis=0, weights=weights)
            variances.append(variance)
        return variances
    
    def get_adaptive_parameters(self):
        """Get current adaptive parameter values for diagnostics."""
        return {
            'Q_std': self.Q_std,
            'R_std': self.R_std, 
            'R_var': self.R_var,
            'innovation_history_length': len(self.innovation_history),
            'recent_innovation_std': np.std(self.innovation_history[-10:]) if len(self.innovation_history) >= 10 else 0.0
        }


In [15]:
# ==============================================================================
# MAIN SIMULATION: EKF vs PF COMPARISON
# ==============================================================================

if __name__ == "__main__":
    # --- Simulation settings ---
    n_steps = 1000
    dt = 0.01
    t_history = np.linspace(0, (n_steps-1) * dt, n_steps)

    process_noise_type = 'laplacian'  # Laplacian noise for better numerical stability
    process_noise_std = 0.5  # Noise level

    # --- True system init ---
    x_true = np.zeros((n_steps, 3))
    x_true[0] = [1.0, 1.0, 1.0]  # Initial conditions for Lorenz system
    u = 0.0

    # --- RHONN config ---
    num_neurons = 3  # Three states for Lorenz system
    num_features = 10  # Feature vector size for 3 states
    num_weights_per_neuron = num_features

    # --- Common initial weights for fair comparison ---
    seed = np.random.randint(0, 2**32-1)  # Random seed for each run
    np.random.seed(seed)  # For reproducibility
    common_initial_weights = [np.random.uniform(-0.1, 0.1, num_weights_per_neuron) for _ in range(num_neurons)]
    print("Common Initial Weights:")
    for i, w in enumerate(common_initial_weights):
        print(f"  Neuron {i}: {w}")

    # --- EKF Trainer ---
    ekf_trainer = EKF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        initial_weights=common_initial_weights,
        Q_init=5e-4, R_init=1e-3, P_init=5.0, eta=1.0
    )
    x_hat_ekf = np.zeros((n_steps, 3))
    x_hat_ekf[0] = x_true[0]

    # --- Particle Filter Trainer ---
    n_particles = 5000  # Increased particles for better performance
    pf_trainer = PF_RHONN_Trainer(
        num_neurons, num_weights_per_neuron,
        n_particles=n_particles,
        initial_weights=common_initial_weights,
        # Q_std=1.0, R_std=0.2,  # Tuned parameters
        Q_std=1.0, R_std=0.67,  # Tuned parameters
        ess_threshold=n_particles / 2  # More frequent resampling
    )
    x_hat_pf = np.zeros((n_steps, 3))
    x_hat_pf[0] = x_true[0]

    print("\n🚀 Starting EKF vs PF simulation...")
    for k in range(n_steps - 1):
        # ---- 1) True system evolution ----
        x_true[k+1] = plant(x_true[k], u, dt, process_noise_type, process_noise_std)

        # ---- 2) EKF update ----
        ekf_trainer.update(chi_kp1=x_true[k+1], x_hat_previous=x_hat_ekf[k])

        # Predict next state using updated EKF weights
        x_hat_ekf[k+1, 0] = RHONN_predict(x_hat_ekf[k], ekf_trainer.weights[0])  # x
        x_hat_ekf[k+1, 1] = RHONN_predict(x_hat_ekf[k], ekf_trainer.weights[1])  # y
        x_hat_ekf[k+1, 2] = RHONN_predict(x_hat_ekf[k], ekf_trainer.weights[2])  # z

        # ---- 3) PF update ----
        pf_trainer.update(chi_kp1=x_true[k+1], x_hat_previous=x_hat_pf[k])

        # Get updated weight estimates from PF
        pf_weight_estimates = pf_trainer.get_estimate()
        
        # Predict next state using updated PF weights
        x_hat_pf[k+1, 0] = RHONN_predict(x_hat_pf[k], pf_weight_estimates[0])   # x
        x_hat_pf[k+1, 1] = RHONN_predict(x_hat_pf[k], pf_weight_estimates[1])   # y
        x_hat_pf[k+1, 2] = RHONN_predict(x_hat_pf[k], pf_weight_estimates[2])   # z

        # if k % 200 == 0:
        #     print(f"📊 Simulation progress: {k/n_steps*100:.1f}%")

    print("✅ Simulation completed successfully!")

Common Initial Weights:
  Neuron 0: [-0.09080019  0.05959007  0.0996483  -0.02472782 -0.01422516 -0.03428421
  0.05604512 -0.06221549  0.09810259  0.04112377]
  Neuron 1: [-0.00064106  0.06165492 -0.07344901 -0.05077609  0.01349486  0.08471915
 -0.04218987  0.04854933  0.0584121  -0.04423992]
  Neuron 2: [ 0.02515663  0.05082142 -0.00717056 -0.0289426  -0.0316022   0.04465117
  0.05140224 -0.03412474  0.04258173 -0.00261499]

🚀 Starting EKF vs PF simulation...
✅ Simulation completed successfully!


In [16]:
# ==============================================================================
# RESULTS AND VISUALIZATION: EKF vs PF PERFORMANCE
# ==============================================================================
mse_x1_ekf = np.mean((x_true[:, 0] - x_hat_ekf[:, 0])**2)  # x
mse_x2_ekf = np.mean((x_true[:, 1] - x_hat_ekf[:, 1])**2)  # y
mse_x3_ekf = np.mean((x_true[:, 2] - x_hat_ekf[:, 2])**2)  # z

mse_x1_pf = np.mean((x_true[:, 0] - x_hat_pf[:, 0])**2)   # x
mse_x2_pf = np.mean((x_true[:, 1] - x_hat_pf[:, 1])**2)   # y
mse_x3_pf = np.mean((x_true[:, 2] - x_hat_pf[:, 2])**2)   # z

print(f"\nFinal EKF-RHONN Weights:")
for i in range(3):
    print(f"  Neuron {i+1} (x{i+1}): {ekf_trainer.weights[i]}")

print(f"\nFinal PF-RHONN Weight Estimates:")
pf_estimates = pf_trainer.get_estimate()
for i in range(3):
    print(f"  Neuron {i+1} (x{i+1}): {pf_estimates[i]}")

# Display adaptive parameter diagnostics
adaptive_params = pf_trainer.get_adaptive_parameters()
print(f"\n--- Adaptive PF Parameter Values ---")
print(f"Final Q_std (process noise): {adaptive_params['Q_std']:.6f}")
print(f"Final R_std (measurement noise): {adaptive_params['R_std']:.6f}")
print(f"Innovation history length: {adaptive_params['innovation_history_length']}")
print(f"Recent innovation std: {adaptive_params['recent_innovation_std']:.6f}")

print("\n--- Performance Comparison (MSE) for Lorenz System ---")
print(f"EKF MSE x1 (x):         {mse_x1_ekf:.6f}")
print(f"EKF MSE x2 (y):         {mse_x2_ekf:.6f}")
print(f"EKF MSE x3 (z):         {mse_x3_ekf:.6f}")
print(f"PF  MSE x1 (x):         {mse_x1_pf:.6f}")
print(f"PF  MSE x2 (y):         {mse_x2_pf:.6f}")
print(f"PF  MSE x3 (z):         {mse_x3_pf:.6f}")

states_info = [
    {'idx': 0, 'var': 'x', 'desc': 'Lorenz X State', 'y_label': 'X Value',
     'chi': 'χ₁ (True x)', 'x': 'x₁ (Est. x)'},
    {'idx': 1, 'var': 'y', 'desc': 'Lorenz Y State', 'y_label': 'Y Value',
     'chi': 'χ₂ (True y)', 'x': 'x₂ (Est. y)'},
    {'idx': 2, 'var': 'z', 'desc': 'Lorenz Z State', 'y_label': 'Z Value',
     'chi': 'χ₃ (True z)', 'x': 'x₃ (Est. z)'}
]

for state_info in states_info:
    i = state_info['idx']
    trace_plant = go.Scatter(x=t_history, y=x_true[:, i], mode='lines',
                            name=state_info['chi'], line=dict(color='black', width=2))
    trace_pf = go.Scatter(x=t_history, y=x_hat_pf[:, i], mode='lines',
                        name=f"{state_info['x']} (PF)", line=dict(dash='dot'))
    trace_ekf = go.Scatter(x=t_history, y=x_hat_ekf[:, i], mode='lines',
                        name=f"{state_info['x']} (EKF)", line=dict(dash='dash'))

    fig = go.Figure([trace_plant, trace_pf, trace_ekf])
    fig.update_layout(
        title=f'Lorenz System RHONN Identification for {state_info["var"]} ({state_info["desc"]})',
        xaxis_title='Time (s)',
        yaxis_title=state_info['y_label'],
        legend=dict(x=0, y=1, orientation='h'),
        font=dict(size=12),
        plot_bgcolor='white',
        paper_bgcolor='white'
    )
    fig.show()

# Errors for all three states
error_x1_ekf = x_true[:, 0] - x_hat_ekf[:, 0]
error_x2_ekf = x_true[:, 1] - x_hat_ekf[:, 1]
error_x3_ekf = x_true[:, 2] - x_hat_ekf[:, 2]

error_x1_pf = x_true[:, 0] - x_hat_pf[:, 0]
error_x2_pf = x_true[:, 1] - x_hat_pf[:, 1]
error_x3_pf = x_true[:, 2] - x_hat_pf[:, 2]

fig2 = go.Figure()
fig2.add_trace(go.Scatter(x=t_history, y=error_x1_ekf, mode='lines',
                        name=f'EKF Error x (MSE={mse_x1_ekf:.6f})', opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_x1_pf, mode='lines',
                        name=f'PF Error x (MSE={mse_x1_pf:.6f})', opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_x2_ekf, mode='lines',
                        name=f'EKF Error y (MSE={mse_x2_ekf:.6f})', opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_x2_pf, mode='lines',
                        name=f'PF Error y (MSE={mse_x2_pf:.6f})', opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_x3_ekf, mode='lines',
                        name=f'EKF Error z (MSE={mse_x3_ekf:.6f})', opacity=0.7))
fig2.add_trace(go.Scatter(x=t_history, y=error_x3_pf, mode='lines',
                        name=f'PF Error z (MSE={mse_x3_pf:.6f})', opacity=0.7))
fig2.update_layout(
    title='Lorenz System Identification Errors',
    xaxis_title='Time (s)',
    yaxis_title='Error',
    legend=dict(x=0, y=1, orientation='h'),
    font=dict(size=12),
    plot_bgcolor='white',
    paper_bgcolor='white'
)
fig2.show()

# 3D Phase space plot
fig3d = go.Figure()
fig3d.add_trace(go.Scatter3d(x=x_true[:, 0], y=x_true[:, 1], z=x_true[:, 2],
                           mode='lines', name='True Lorenz Attractor',
                           line=dict(color='black', width=3)))
fig3d.add_trace(go.Scatter3d(x=x_hat_ekf[:, 0], y=x_hat_ekf[:, 1], z=x_hat_ekf[:, 2],
                           mode='lines', name='EKF Estimation',
                           line=dict(color='red', width=2, dash='dash')))
fig3d.add_trace(go.Scatter3d(x=x_hat_pf[:, 0], y=x_hat_pf[:, 1], z=x_hat_pf[:, 2],
                           mode='lines', name='PF Estimation',
                           line=dict(color='blue', width=2, dash='dot')))
fig3d.update_layout(
    title='Lorenz System - 3D Phase Space Comparison',
    scene=dict(
        xaxis_title='X',
        yaxis_title='Y',
        zaxis_title='Z'
    ),
    font=dict(size=12)
)
fig3d.show()


Final EKF-RHONN Weights:
  Neuron 1 (x1): [ 20.87432048   1.7550958  -23.28531298   1.30795558  11.40433807
   2.0840788  -27.92153225   1.39409271   2.47757075  10.90120469]
  Neuron 2 (x2): [  6.44688183  -5.65675284 -24.43980699  -0.56776395   5.57175391
   9.46603651 -12.43008234   2.02770024   7.02089264   8.75503007]
  Neuron 3 (x3): [ 1.3693788  11.28485827 27.66314902  0.32957074  1.73599433 -9.2176647
  3.88270427  2.23496917  3.6766828  -2.41805394]

Final PF-RHONN Weight Estimates:
  Neuron 1 (x1): [ -2.27291694  57.56494433  12.80228134   4.18849984  -3.69174081
 -55.01148509   6.75771992  -1.50080853   2.72059095 -25.45319419]
  Neuron 2 (x2): [ 14.58321925  40.57826862   0.92564058 -15.27444289  17.04705767
 -29.62330515 -32.71216211   4.09218269 -45.27701844  35.7933192 ]
  Neuron 3 (x3): [  9.65052595  -7.0286803   26.80501491   5.42081456   7.16011068
   8.45677055 -10.54792188  -5.04593102  15.87317035 -13.85343821]

--- Adaptive PF Parameter Values ---
Final Q_std (

### Recommended seeds
* 2454815412